In [0]:

SRC = "workspace.bronze.fhir_bundles_raw"
TGT = "workspace.silver.patients"
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")


In [0]:
display(spark.sql(f"""
                  SELECT 
                  get_json_object(e,'$.resource.id' )       as patient_id,
                  get_json_object(e,'$.resource.name[0].given[0]' )     as patient_first_name,
                  get_json_object(e,'$.resource.name[0].family' )     as patient_last_name,
                  get_json_object(e,'$.resource.gender' )     as patient_gender   ,   
                  Cast (get_json_object(e,'$.resource.birthDate') as date)     as patient_birthDate,
                  source_file

                  FROM {SRC}
                  LATERAL VIEW explode(from_json(get_json_object(raw_json,'$.entry'),'array<string>')) t AS e
                  WHERE get_json_object(e,'$.resource.resourceType')='Patient'
                  LIMIT 5"""))